# Auditing a hiring model

This notebook goes with the Detailed lesson **Responsible AI: Principles, Fairness and Accountability**, section 5.

We compute selection rates, the four-fifths rule and true positive rates for two groups, before and after a fix. Run each cell in order with **Shift + Enter**.

In [ ]:
import matplotlib.pyplot as plt

def audit(name, groups):
    print(name)
    rates = {}
    for g, d in groups.items():
        rates[g] = d["shortlisted"] / d["applicants"]
        line = f"  Group {g}: {d['shortlisted']} of {d['applicants']} shortlisted, selection rate {rates[g]:.0%}"
        if "qualified" in d:
            line += f", true positive rate {d['qualified_shortlisted'] / d['qualified']:.0%}"
        print(line)
    ratio = min(rates.values()) / max(rates.values())
    print(f"  Disparate impact ratio {ratio:.2f}: {'passes' if ratio >= 0.8 else 'FLAGGED'} under the four-fifths rule")
    return rates, ratio

## 1. Before the fix

In [ ]:
before = {
    "A": {"applicants": 200, "shortlisted": 60, "qualified": 80, "qualified_shortlisted": 52},
    "B": {"applicants": 150, "shortlisted": 27, "qualified": 60, "qualified_shortlisted": 24},
}
rates_before, ratio_before = audit("Original model", before)

## 2. After removing the proxy feature and retraining

In [ ]:
after = {
    "A": {"applicants": 200, "shortlisted": 58},
    "B": {"applicants": 150, "shortlisted": 36},
}
rates_after, ratio_after = audit("Retrained model", after)

## 3. Comparing the two

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
x = [0, 1]
ax.bar([i - 0.2 for i in x], [rates_before["A"], rates_after["A"]], width=0.4, label="Group A")
ax.bar([i + 0.2 for i in x], [rates_before["B"], rates_after["B"]], width=0.4, label="Group B")
ax.set_xticks(x, [f"original (ratio {ratio_before:.2f})", f"retrained (ratio {ratio_after:.2f})"])
ax.set_ylabel("selection rate"); ax.legend(); ax.set_title("Selection rates by group")
plt.show()

## 4. Try it yourself

- Change the numbers in `after`. What is the smallest number of Group B shortlists, out of 150, that passes the four-fifths rule when Group A's rate is 29%?
- Add a third group and extend the audit to compare every group with the highest rate.